# NBA Data Warehouse — Analysis Playground
**BSAN 6060 | Faisal Alessa · Jake Bonavia · Dante Shoghanian**

No Google Drive needed. No setup. Just **Runtime → Run All** and start exploring.

---

| Section | What it does |
|---|---|
| ① Setup | Loads the database from GitHub |
| ② Schema | Browse all 11 tables |
| ③ Player Value | Undervalued / Overvalued players by season |
| ④ Team Analysis | Net Rating, Win %, Offense vs Defense |
| ⑤ Player Lookup | Search any player across all 4 seasons |
| ⑥ Playoff Performers | Who holds up when it matters |
| ⑦ Hustle Stats | Hidden contributors beyond the box score |
| ⑧ Free Agent Targets | Match undervalued players to a team's need |
| ⑨ Custom Query | Write your own SQL |

---
**Value tiers:** 🔵 Undervalued (top 33%) · ⬜ Fair Value · 🔴 Overvalued (bottom 33%)

> Max contract players ($40M+) are excluded from tiers — dividing by a $45M salary always scores low. That is a model limitation, not a basketball judgment.

---
## ① Setup — Run This First

In [ ]:
# ── Load database from GitHub ─────────────────────────────────────────────────
# Clones the repo and connects to the database
# Takes about 30 seconds on first run

import os

REPO = 'nba-analytics-pipeline'

if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/faisalalessa123/nba-analytics-pipeline.git
    print('Repo cloned.')
else:
    !git -C /content/{REPO} pull
    print('Repo already present — pulled latest.')

# ── Connect ───────────────────────────────────────────────────────────────────
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

conn = sqlite3.connect(f'/content/{REPO}/nba_warehouse.db')

# ── Sanity check ──────────────────────────────────────────────────────────────
tables  = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
seasons = pd.read_sql("SELECT SEASON FROM Season ORDER BY SEASON", conn)
print(f'Connected.')
print(f'Tables:  {list(tables.name)}')
print(f'Seasons: {list(seasons.SEASON)}')

# ── Color constants ───────────────────────────────────────────────────────────
NBA_BLUE     = '#1D428A'
NBA_RED      = '#C8102E'
GOLD         = '#E6AC00'
GREEN        = '#2E7D32'
GRAY         = '#CCCCCC'
TIER_COLORS  = {'Undervalued': NBA_BLUE, 'Fair Value': GRAY, 'Overvalued': NBA_RED}

print('\nReady. Run any section below.')

---
## ② Schema Browser

In [ ]:
# ── All tables and columns ────────────────────────────────────────────────────
# Run this whenever you need to know what columns are available for a query

for tbl in tables.name:
    cols = pd.read_sql(f'PRAGMA table_info({tbl})', conn)
    print(f"\n{'─'*55}")
    print(f'  {tbl}  ({len(cols)} columns)')
    print(f"{'─'*55}")
    for _, r in cols.iterrows():
        tag = ' ← PK' if r['pk'] else ''
        print(f"  {r['type']:<8}  {r['name']}{tag}")

---
## ③ Player Value Scores
Change `SEASON` and `TIER` to explore any combination.

In [ ]:
# ── Top players by value tier ─────────────────────────────────────────────────
# Composite Value Score = PIE + Net Rating + TS% (each normalized 0-1 and divided by salary)
# Higher score = more performance per dollar

SEASON = '2024-25'    # ← '2021-22' | '2022-23' | '2023-24' | '2024-25'
TIER   = 'Undervalued' # ← 'Undervalued' | 'Fair Value' | 'Overvalued'
LIMIT  = 20

value_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.POSITION,
        pvs.TEAM_ABBREVIATION                  AS TEAM,
        pvs.VALUE_TIER,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4)    AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)          AS SALARY_M,
        ROUND(pvs.NET_RATING, 2)               AS NET_RATING,
        ROUND(pvs.PLAYER_IMPACT_ESTIMATE, 3)   AS PIE,
        ROUND(pvs.TRUE_SHOOTING_PCT * 100, 1)  AS TS_PCT,
        pvs.GAMES_PLAYED
    FROM PlayerValueScores pvs
    WHERE pvs.SEASON          = '{SEASON}'
      AND pvs.VALUE_TIER      = '{TIER}'
      AND pvs.SALARY_MILLIONS >= 5
      AND pvs.GAMES_PLAYED    >= 50
    ORDER BY pvs.COMPOSITE_VALUE_SCORE DESC
    LIMIT {LIMIT}
""", conn)

print(f'{TIER} Players — {SEASON} (min $5M, min 50 games)')
print(value_df.to_string(index=False))

In [ ]:
# ── Visualization: Salary vs Composite Value Score ────────────────────────────
# Top-left = undervalued targets (high score, low salary)
# Bottom-right = overpaid (low score, high salary)

SEASON = '2024-25'  # ← change me

scatter_df = pd.read_sql(f"""
    SELECT PLAYER_NAME, VALUE_TIER, SALARY_MILLIONS, COMPOSITE_VALUE_SCORE
    FROM PlayerValueScores
    WHERE SEASON = '{SEASON}' AND SALARY_MILLIONS >= 5 AND GAMES_PLAYED >= 50
""", conn)

fig, ax = plt.subplots(figsize=(12, 7))
for tier, grp in scatter_df.groupby('VALUE_TIER'):
    ax.scatter(grp.SALARY_MILLIONS, grp.COMPOSITE_VALUE_SCORE,
               c=TIER_COLORS.get(tier, GRAY), label=tier,
               alpha=0.75, s=60, edgecolors='white', linewidths=0.4)

# Label top 8 undervalued
top8 = scatter_df[scatter_df.VALUE_TIER=='Undervalued'].nlargest(8,'COMPOSITE_VALUE_SCORE')
for _, r in top8.iterrows():
    ax.annotate(r.PLAYER_NAME.split()[-1], (r.SALARY_MILLIONS, r.COMPOSITE_VALUE_SCORE),
                fontsize=7.5, xytext=(4,4), textcoords='offset points')

ax.set_xlabel('Salary (Millions $)', fontsize=11)
ax.set_ylabel('Composite Value Score', fontsize=11)
ax.set_title(f'Salary vs Value Score — {SEASON}', fontsize=13, fontweight='bold', color='#0D1A33')
ax.legend(title='Value Tier', fontsize=9)
ax.grid(True, alpha=0.2)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

---
## ④ Team Analysis

In [ ]:
# ── All teams ranked by Net Rating ───────────────────────────────────────────
# Net Rating explains 85%+ of Win % variance — strongest single predictor of winning

SEASON = '2024-25'  # ← change me

teams_df = pd.read_sql(f"""
    SELECT
        t.TEAM_ABBREVIATION,
        t.TEAM_NAME,
        ROUND(ts.WIN_PCT * 100, 1)    AS WIN_PCT,
        ROUND(ts.NET_RATING, 2)       AS NET_RATING,
        ROUND(ts.OFFENSIVE_RATING, 2) AS OFF_RTG,
        ROUND(ts.DEFENSIVE_RATING, 2) AS DEF_RTG,
        ROUND(ts.PACE, 1)             AS PACE
    FROM TeamSeasonStats ts
    JOIN Team t ON ts.TEAM_ID = t.TEAM_ID
    WHERE ts.SEASON = '{SEASON}'
    ORDER BY ts.NET_RATING DESC
""", conn)

print(f'Teams by Net Rating — {SEASON}')
print(teams_df.to_string(index=False))

In [ ]:
# ── Visualization: Offensive vs Defensive Rating ──────────────────────────────
# Top-right = contenders  |  Bottom-left = rebuilding
# Top-left = defensive identity  |  Bottom-right = offensive identity
# NOTE: Y-axis is inverted — lower defensive rating = better defense

SEASON = '2024-25'  # ← change me

fig, ax = plt.subplots(figsize=(12, 8))
colors = [NBA_BLUE if nr > 0 else NBA_RED for nr in teams_df.NET_RATING]
ax.scatter(teams_df.OFF_RTG, teams_df.DEF_RTG,
           c=colors, s=100, alpha=0.85, edgecolors='white', linewidths=0.5, zorder=3)

for _, r in teams_df.iterrows():
    ax.annotate(r.TEAM_ABBREVIATION, (r.OFF_RTG, r.DEF_RTG),
                fontsize=8, xytext=(5,3), textcoords='offset points')

ax.axvline(teams_df.OFF_RTG.mean(), color='gray', linestyle='--', lw=1, alpha=0.4)
ax.axhline(teams_df.DEF_RTG.mean(), color='gray', linestyle='--', lw=1, alpha=0.4)
ax.invert_yaxis()  # lower defensive rating = better
ax.set_xlabel('Offensive Rating (higher = better)', fontsize=11)
ax.set_ylabel('Defensive Rating (lower = better)', fontsize=11)
ax.set_title(f'Team Identity — {SEASON}', fontsize=13, fontweight='bold', color='#0D1A33')
ax.legend(handles=[
    mpatches.Patch(color=NBA_BLUE, label='Positive Net Rating'),
    mpatches.Patch(color=NBA_RED,  label='Negative Net Rating')
], fontsize=9)
ax.grid(True, alpha=0.15)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

In [ ]:
# ── Single team roster value breakdown ───────────────────────────────────────
# See how every player on a roster is rated by the model
# Common abbreviations: LAL, GSW, BOS, MIA, LAC, HOU, OKC, PHX, DEN, NYK

TEAM   = 'LAC'      # ← change me
SEASON = '2024-25'  # ← change me

roster_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.POSITION,
        pvs.VALUE_TIER,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4) AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)       AS SALARY_M,
        ROUND(pvs.NET_RATING, 2)            AS NET_RATING,
        ROUND(pvs.PLAYER_IMPACT_ESTIMATE, 3) AS PIE,
        pvs.GAMES_PLAYED
    FROM PlayerValueScores pvs
    WHERE pvs.SEASON            = '{SEASON}'
      AND pvs.TEAM_ABBREVIATION = '{TEAM}'
      AND pvs.SALARY_MILLIONS   >= 5
    ORDER BY pvs.SALARY_MILLIONS DESC
""", conn)

print(f'{TEAM} Roster — {SEASON}')
print(roster_df.to_string(index=False))

---
## ⑤ Player Lookup

In [ ]:
# ── Player across all seasons ─────────────────────────────────────────────────
# Partial names work — 'Caruso', 'Zubac', 'Haliburton', 'Leonard' all work

PLAYER = 'Caruso'  # ← change me

player_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.SEASON,
        pvs.TEAM_ABBREVIATION               AS TEAM,
        pvs.VALUE_TIER,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4)  AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)        AS SALARY_M,
        ROUND(pvs.NET_RATING, 2)             AS NET_RATING,
        ROUND(pvs.PLAYER_IMPACT_ESTIMATE, 3) AS PIE,
        ROUND(pvs.TRUE_SHOOTING_PCT * 100, 1) AS TS_PCT,
        pvs.GAMES_PLAYED
    FROM PlayerValueScores pvs
    WHERE pvs.PLAYER_NAME    LIKE '%{PLAYER}%'
      AND pvs.SALARY_MILLIONS >= 5
    ORDER BY pvs.SEASON
""", conn)

if player_df.empty:
    print(f'No results for "{PLAYER}" — try a different spelling')
else:
    print(player_df.to_string(index=False))

In [ ]:
# ── Head to head comparison ───────────────────────────────────────────────────

PLAYER_A = 'Caruso'   # ← change me
PLAYER_B = 'Payton'   # ← change me
SEASON   = '2024-25'  # ← change me

h2h_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME, pvs.POSITION,
        pvs.TEAM_ABBREVIATION               AS TEAM,
        pvs.VALUE_TIER,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4)  AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)        AS SALARY_M,
        ROUND(pvs.NET_RATING, 2)             AS NET_RATING,
        ROUND(pvs.PLAYER_IMPACT_ESTIMATE, 3) AS PIE,
        ROUND(pvs.TRUE_SHOOTING_PCT * 100, 1) AS TS_PCT,
        pvs.GAMES_PLAYED
    FROM PlayerValueScores pvs
    WHERE pvs.SEASON = '{SEASON}'
      AND (pvs.PLAYER_NAME LIKE '%{PLAYER_A}%'
           OR pvs.PLAYER_NAME LIKE '%{PLAYER_B}%')
    ORDER BY pvs.COMPOSITE_VALUE_SCORE DESC
""", conn)

print(f'Head to Head — {SEASON}')
print(h2h_df.to_string(index=False))

---
## ⑥ Playoff Performers

In [ ]:
# ── Undervalued players who held up in the playoffs ───────────────────────────
# INNER JOIN means only players who actually appeared in the playoffs are returned
# NET_RATING > 0 means their team outscored opponents with them on the floor

SEASON = '2024-25'  # ← change me

playoff_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.POSITION,
        pvs.TEAM_ABBREVIATION               AS TEAM,
        ROUND(pvs.SALARY_MILLIONS, 1)        AS SALARY_M,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4)  AS REG_SCORE,
        ROUND(pvs.NET_RATING, 2)             AS REG_NET_RTG,
        ROUND(pps.NET_RATING, 2)             AS PLAYOFF_NET_RTG,
        ROUND(pps.PLAYER_IMPACT_ESTIMATE, 3) AS PLAYOFF_PIE,
        ROUND(pps.TRUE_SHOOTING_PCT * 100, 1) AS PLAYOFF_TS_PCT,
        pps.GAMES_PLAYED                     AS PLAYOFF_GP
    FROM PlayerValueScores pvs
    JOIN PlayoffPlayerStats pps
        ON pvs.PLAYER_ID = pps.PLAYER_ID
        AND pvs.SEASON   = pps.SEASON
    WHERE pvs.SEASON          = '{SEASON}'
      AND pvs.VALUE_TIER      = 'Undervalued'
      AND pvs.SALARY_MILLIONS >= 5
      AND pvs.GAMES_PLAYED    >= 50
      AND pps.NET_RATING       > 0
    ORDER BY pps.NET_RATING DESC
    LIMIT 15
""", conn)

print(f'Undervalued Players with Positive Playoff Net Rating — {SEASON}')
print(playoff_df.to_string(index=False))

---
## ⑦ Hustle Stats

In [ ]:
# ── Top hustle players ────────────────────────────────────────────────────────
# Hustle Score = deflections + contested shots + charges drawn + box outs
# These never show up in the traditional box score

SEASON = '2024-25'  # ← change me

hustle_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.POSITION,
        pvs.TEAM_ABBREVIATION            AS TEAM,
        pvs.VALUE_TIER,
        ROUND(pvs.SALARY_MILLIONS, 1)     AS SALARY_M,
        ROUND(pss.DEFLECTIONS, 2)         AS DEFLECTIONS,
        ROUND(pss.CONTESTED_SHOTS, 2)     AS CONTESTED_SHOTS,
        ROUND(pss.CHARGES_DRAWN, 2)       AS CHARGES_DRAWN,
        ROUND(pss.BOX_OUTS, 2)            AS BOX_OUTS,
        ROUND(pss.DEFLECTIONS + pss.CONTESTED_SHOTS
            + pss.CHARGES_DRAWN + pss.BOX_OUTS, 2) AS HUSTLE_SCORE
    FROM PlayerValueScores pvs
    JOIN PlayerSeasonStats pss
        ON pvs.PLAYER_ID = pss.PLAYER_ID
        AND pvs.SEASON   = pss.SEASON
    WHERE pvs.SEASON          = '{SEASON}'
      AND pvs.SALARY_MILLIONS >= 5
      AND pvs.GAMES_PLAYED    >= 50
    ORDER BY HUSTLE_SCORE DESC
    LIMIT 20
""", conn)

print(f'Top Hustle Players — {SEASON}')
print(hustle_df.to_string(index=False))

---
## ⑧ Free Agent Targets by Team Need

In [ ]:
# ── Find undervalued targets for a specific team ──────────────────────────────
# Filters to players outside the target team (free agent / trade candidates)
# who are undervalued, appeared in the playoffs, and play the needed position
#
# Positions: 'G' | 'F' | 'C' | 'G-F' | 'F-G' | 'F-C' | 'C-F'

TARGET_TEAM = 'LAC'              # ← team you are targeting for
POSITIONS   = "'G','G-F','F-G'"  # ← positions of need (comma separated, keep quotes)
MAX_SALARY  = 20                  # ← max contract value in millions
SEASON      = '2024-25'

targets_df = pd.read_sql(f"""
    SELECT
        pvs.PLAYER_NAME,
        pvs.POSITION,
        pvs.TEAM_ABBREVIATION               AS CURRENT_TEAM,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4)  AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)        AS SALARY_M,
        ROUND(pvs.NET_RATING, 2)             AS NET_RATING,
        ROUND(pss.DEFENSIVE_RATING, 2)       AS DEF_RATING,
        ROUND(pss.DEFLECTIONS, 2)            AS DEFLECTIONS,
        ROUND(pss.CONTESTED_SHOTS, 2)        AS CONTESTED_SHOTS,
        ROUND(pps.NET_RATING, 2)             AS PLAYOFF_NET_RTG
    FROM PlayerValueScores pvs
    JOIN PlayerSeasonStats pss
        ON pvs.PLAYER_ID = pss.PLAYER_ID AND pvs.SEASON = pss.SEASON
    JOIN PlayoffPlayerStats pps
        ON pvs.PLAYER_ID = pps.PLAYER_ID AND pvs.SEASON = pps.SEASON
    WHERE pvs.SEASON             = '{SEASON}'
      AND pvs.VALUE_TIER         = 'Undervalued'
      AND pvs.TEAM_ABBREVIATION != '{TARGET_TEAM}'
      AND pvs.SALARY_MILLIONS    BETWEEN 5 AND {MAX_SALARY}
      AND pvs.GAMES_PLAYED       >= 50
      AND pvs.POSITION           IN ({POSITIONS})
      AND pps.NET_RATING          > 0
    ORDER BY pps.NET_RATING DESC
    LIMIT 15
""", conn)

print(f'Undervalued Targets for {TARGET_TEAM} — {SEASON}')
print(f'Position need: {POSITIONS} | Max salary: ${MAX_SALARY}M')
print(targets_df.to_string(index=False))

---
## ⑨ Custom Query
Write any SQL you want against the database.

In [ ]:
# ── Write your own SQL here ───────────────────────────────────────────────────
# All 11 tables are available:
#   Player, Team, Season (dimensions)
#   PlayerSeasonStats, PlayerShootingStats, PlayerSalary,
#   PlayerValueScores, PlayoffPlayerStats, PlayoffPlayerDefenseStats,
#   PlayoffTeamStats, TeamSeasonStats (facts)

custom_df = pd.read_sql("""
    SELECT
        pvs.PLAYER_NAME,
        pvs.SEASON,
        pvs.TEAM_ABBREVIATION,
        pvs.VALUE_TIER,
        ROUND(pvs.COMPOSITE_VALUE_SCORE, 4) AS COMPOSITE_SCORE,
        ROUND(pvs.SALARY_MILLIONS, 1)       AS SALARY_M
    FROM PlayerValueScores pvs
    WHERE pvs.PLAYER_NAME LIKE '%Leonard%'
    ORDER BY pvs.SEASON
""", conn)

print(custom_df.to_string(index=False))